In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os

# 1. Setup Device (uses GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Data Transformations (Crucial for CV)
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),              # Resize shortest side to 256
        transforms.CenterCrop(224),          # Crop the center 224x224 (keeps logo/login box)
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1), # Add slight noise (robustness)
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 3. Load Data
data_dir = 'datasets' # Ensure you have this folder structure
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=4)
               for x in ['train', 'val']}

model = models.resnet18(pretrained=True)

# 1. First, freeze everything
for param in model.parameters():
    param.requires_grad = False

# 2. Unfreeze the LAST block (layer4) and the Fully Connected layer (fc)
# This lets the model learn high-level webpage features
for param in model.layer4.parameters():
    param.requires_grad = True

# 3. Re-initialize the classifier
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5), # Helps prevent overfitting
    nn.Linear(num_ftrs, 2)
)

model = model.to(device)

# 4. Use a smaller learning rate for fine-tuning
criterion = nn.CrossEntropyLoss()
# Optimize only the parameters we unfroze (layer4 and fc)
params_to_update = []
for name, param in model.named_parameters():
    if param.requires_grad == True:
        params_to_update.append(param)

optimizer = optim.Adam(params_to_update, lr=0.0001) # Lower LR (1e-4) is safer for fine-tuning

# 6. Training Loop (Simplified)
def train_model(model, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    return model

Using device: cuda


In [9]:
trained_model = train_model(model, criterion, optimizer, num_epochs=5)
torch.save(trained_model.state_dict(), 'phishing_visual_model.pth')

Epoch 1/5
----------
train Loss: 0.6517 Acc: 0.6494
val Loss: 0.6125 Acc: 0.6667
Epoch 2/5
----------
train Loss: 0.4764 Acc: 0.7606
val Loss: 0.5895 Acc: 0.7040
Epoch 3/5
----------
train Loss: 0.3823 Acc: 0.8132
val Loss: 0.6514 Acc: 0.6868
Epoch 4/5
----------
train Loss: 0.3159 Acc: 0.8473
val Loss: 0.6525 Acc: 0.7011
Epoch 5/5
----------
train Loss: 0.2876 Acc: 0.8666
val Loss: 0.7412 Acc: 0.7126


In [18]:
import numpy
print(numpy.__version__)    

2.1.3


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Define the Grad-CAM Hook
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        
        # Hook to capture gradients
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activation = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, class_idx=None):
        # Forward pass
        output = self.model(x)
        if class_idx is None:
            class_idx = torch.argmax(output)

        # Backward pass
        self.model.zero_grad()
        class_loss = output[0, class_idx]
        class_loss.backward()

        # Generate Heatmap
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activation = self.activation[0]
        
        for i in range(activation.shape[0]):
            activation[i, :, :] *= pooled_gradients[i]
            
        heatmap = torch.mean(activation, dim=0).cpu().detach().numpy()
        heatmap = np.maximum(heatmap, 0) # ReLU
        heatmap /= torch.max(torch.from_numpy(heatmap)) # Normalize
        return heatmap

# 2. Function to Overlay Heatmap on Image
def show_cam_on_image(img_tensor, heatmap):
    # Convert tensor back to image format (H, W, C)
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    
    # Denormalize (reverse the normalization we did in training)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)

    # Resize heatmap to match image size
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    
    # Create color map (JET is the classic heatmap look)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = np.float32(heatmap_colored) / 255
    
    # Overlay: 60% original image + 40% heatmap
    overlay = 0.6 * img + 0.4 * heatmap_colored
    overlay = np.clip(overlay, 0, 1) # Ensure valid pixel range

    # Plot
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Original Screenshot")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title("Model Attention (Grad-CAM)")
    plt.axis('off')
    plt.show()

# 3. Run it on a Test Image
# Get a single batch of images
dataiter = iter(dataloaders['val'])
images, labels = next(dataiter)

# Select one image from the batch (e.g., index 0)
test_image = images[0].unsqueeze(0).to(device) # Add batch dimension

# Initialize Grad-CAM on the LAST convolutional layer of ResNet
# For ResNet18, the last spatial layer is 'layer4'
grad_cam = GradCAM(model, model.layer4)

# Generate and Show
print(f"True Label: {['Legitimate', 'Phishing'][labels[0]]}")
heatmap = grad_cam(test_image)
show_cam_on_image(images[0], heatmap)

ModuleNotFoundError: No module named 'cv2'